# k5 — Stage 2: xếp hạng lại top-N (ColBERT BGE-M3 / cross-encoder) + bài nộp (GPU T4 ×2)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`, `mir2-runs` (có `qtrans/queries_*.parquet`).
`METHODS`: `colbert` (tốt nhất ở dự án cũ, 0,1674) và/hoặc `cross` (`BAAI/bge-reranker-v2-m3`). `TOP_N` phải lớn hơn
k_chunk (F2 tính theo TẬP: xếp hạng lại trong đúng tập nộp không đổi điểm). Dự án cũ: ColBERT top-400 ~100 phút / 2 GPU.
**Đầu ra:** `runs/<run>_<method>.parquet`, `submissions/*.zip` (doc_id số nguyên, tự giảm k_chunk nếu > 99 MB).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k5-v1 (15/10/2026)'
RUN = 'rrf9'
METHODS = ['colbert', 'cross']
TOP_N = 400
K_DOC, K_CHUNK = 200, 185
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks', 'qtrans'))

In [ ]:
# (2) Xếp hạng lại trên 2 GPU (chia theo câu hỏi), ghi vào /kaggle/working/runs
RUNS_IN = locate('runs') / 'runs'
OUT = f'{K.work}/runs'
for m in METHODS:
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.rerank --config {K.cfg} --run {RUNS_IN}/{RUN}.parquet '
         f'--method {m} --top-n {TOP_N} --shard {g}/2 --device cuda:0 --out {OUT}' for g in range(2)], f'rerank_{m}')
    sh(f'python -m mirret.rerank --config {K.cfg} --run {RUNS_IN}/{RUN}.parquet --method {m} --top-n {TOP_N} '
       f'--shard 0/2 --out {OUT}')                    # gộp 2 phần

In [ ]:
# (3) Bài nộp: run gộp (không rerank) + từng run đã rerank
K.set({'retrieve.out_dir': OUT})
shutil.copy(f'{RUNS_IN}/{RUN}.parquet', f'{OUT}/{RUN}.parquet')
for name in [RUN] + [f'{RUN}_{m}' for m in METHODS]:
    sh(f'python -m mirret.submit --config {K.cfg} --run {OUT}/{name}.parquet --k-doc {K_DOC} --k-chunk {K_CHUNK}')
report('k5')